In [ ]:
# M2: RUL ESTIMATOR
# Gradient Boosted Regressor — predicts remaining useful life in hours
# Top 1% additions: confidence intervals + failure-mode-aware training

import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_predict

session = get_active_session()
session.sql("USE DATABASE FAILURE_GENOME_DB").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
print("Session ready.")

In [ ]:
# Load RUL training view (only degrading assets with known failure dates)
df = session.table("FAILURE_GENOME_DB.ML_FEATURES.TRAIN_RUL").to_pandas()
print(f"Loaded {len(df)} rows")
print(f"\nFailure modes in data:\n{df['FAILURE_MODE'].value_counts()}")
print(f"\nRUL range: {df['HOURS_TO_FAILURE'].min():.0f} - {df['HOURS_TO_FAILURE'].max():.0f} hours")

In [ ]:
# Feature columns
exclude_cols = ['ASSET_ID', 'TIMESTAMP', 'HOURS_TO_FAILURE', 'FAILURE_MODE', 'DEGRADATION_STAGE']
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Add failure_mode as encoded feature (failure-mode-aware)
from sklearn.preprocessing import LabelEncoder
le_mode = LabelEncoder()
df['FAILURE_MODE_ENCODED'] = le_mode.fit_transform(df['FAILURE_MODE'])
feature_cols.append('FAILURE_MODE_ENCODED')

print(f"Using {len(feature_cols)} features (including failure_mode as input)")

# Handle nulls and infinities
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# Target: hours to failure
target_col = 'HOURS_TO_FAILURE'

# Time-based split (80/20)
df = df.sort_values('TIMESTAMP')
split_idx = int(len(df) * 0.8)
X_train, X_test = df.iloc[:split_idx][feature_cols].values, df.iloc[split_idx:][feature_cols].values
y_train, y_test = df.iloc[:split_idx][target_col].values, df.iloc[split_idx:][target_col].values
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Target range - Train: {y_train.min():.0f}-{y_train.max():.0f}h | Test: {y_test.min():.0f}-{y_test.max():.0f}h")

In [ ]:
# Main model: predicts median RUL (quantile=0.5)
model_median = XGBRegressor(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
model_median.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Median RUL model trained.")

In [ ]:
# Lower bound (5th percentile) — optimistic estimate
model_lower = XGBRegressor(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    objective='reg:quantileerror',
    quantile_alpha=0.05,
    random_state=42,
    n_jobs=-1
)
model_lower.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# Upper bound (95th percentile) — conservative estimate
model_upper = XGBRegressor(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    objective='reg:quantileerror',
    quantile_alpha=0.95,
    random_state=42,
    n_jobs=-1
)
model_upper.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Confidence interval models trained (5th & 95th percentile).")

In [ ]:
# Predictions
y_pred = model_median.predict(X_test)
y_lower = model_lower.predict(X_test)
y_upper = model_upper.predict(X_test)

# Clip negative predictions to 0 (can't have negative RUL)
y_pred = np.maximum(y_pred, 0)
y_lower = np.maximum(y_lower, 0)
y_upper = np.maximum(y_upper, 0)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Coverage: % of actuals within confidence interval
coverage = np.mean((y_test >= y_lower) & (y_test <= y_upper)) * 100

print(f"=== RUL Estimation Results ===")
print(f"RMSE:     {rmse:.2f} hours")
print(f"MAE:      {mae:.2f} hours")
print(f"R²:       {r2:.4f}")
print(f"90% CI Coverage: {coverage:.1f}% (target: ≥90%)")
print(f"\n=== Sample Predictions (first 10) ===")
sample = pd.DataFrame({
    'Actual_RUL': y_test[:10].round(1),
    'Predicted_RUL': y_pred[:10].round(1),
    'Lower_CI': y_lower[:10].round(1),
    'Upper_CI': y_upper[:10].round(1),
    'Error': (y_pred[:10] - y_test[:10]).round(1)
})
print(sample.to_string(index=False))

In [ ]:
# Top 15 most important features for RUL prediction
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_median.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print("Top 15 Features for RUL Prediction:")
print(importance.to_string(index=False))

In [ ]:
# Test inference from registry (verifies artifact loading works)
from snowflake.ml.registry import Registry

reg_model = Registry(session=session, database_name="FAILURE_GENOME_DB", schema_name="ML_MODELS").get_model("RUL_ESTIMATOR").version("V1")
test_sample = pd.DataFrame(X_test[:5], columns=feature_cols)
result = reg_model.run(test_sample, function_name="predict")
print("Inference test (with confidence intervals):")
print(result)
print(f"\nActual RUL: {y_test[:5]}")

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

class RULEstimator(custom_model.CustomModel):
    def __init__(self, context):
        super().__init__(context)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import numpy as np

        cols = [c for c in input_df.columns if c not in ['ASSET_ID', 'TIMESTAMP', 'HOURS_TO_FAILURE', 'FAILURE_MODE', 'DEGRADATION_STAGE']]
        X = input_df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).values

        pred_median = np.maximum(model_median.predict(X), 0)
        pred_lower = np.maximum(model_lower.predict(X), 0)
        pred_upper = np.maximum(model_upper.predict(X), 0)

        return pd.DataFrame({
            'PREDICTED_RUL_HOURS': pred_median.round(1),
            'RUL_LOWER_CI': pred_lower.round(1),
            'RUL_UPPER_CI': pred_upper.round(1),
            'CONFIDENCE_WIDTH': (pred_upper - pred_lower).round(1)
        })

# Register
reg = Registry(session=session, database_name="FAILURE_GENOME_DB", schema_name="ML_MODELS")

try:
    reg.delete_model("RUL_ESTIMATOR")
    print("Deleted existing model.")
except:
    pass

rul_model = RULEstimator(custom_model.ModelContext())
sample_input = pd.DataFrame(np.zeros((1, len(feature_cols))), columns=feature_cols)

mv = reg.log_model(
    rul_model,
    model_name="RUL_ESTIMATOR",
    version_name="V1",
    metrics={"rmse": float(rmse), "mae": float(mae), "r2": float(r2), "ci_coverage": float(coverage)},
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    comment=f"Gradient Boosted RUL estimator with 90% CI. RMSE={rmse:.2f}h, MAE={mae:.2f}h"
)
print(f"\nModel registered: RUL_ESTIMATOR V1 | RMSE: {rmse:.2f}h | MAE: {mae:.2f}h | R²: {r2:.4f}")

In [ ]:
# Test inference
reg_model = reg.get_model("RUL_ESTIMATOR").version("V1")
test_sample = pd.DataFrame(X_test[:5], columns=feature_cols)
result = reg_model.run(test_sample, function_name="predict")
print("Inference test (with confidence intervals):")
print(result)
print(f"\nActual RUL: {y_test[:5].round(1)}")